In [111]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [112]:
df= pd.read_csv("../data/retail_sales_dataset.csv")

In [113]:
df.head()

,Transaction ID,Date,Customer ID,Gender,Age,Product Category,Quantity,Price per Unit,Total Amount
0,1,2023-11-24,CUST001,Male,34,Beauty,3,50,150
1,2,2023-02-27,CUST002,Female,26,Clothing,2,500,1000
2,3,2023-01-13,CUST003,Male,50,Electronics,1,30,30
3,4,2023-05-21,CUST004,Male,37,Clothing,1,500,500
4,5,2023-05-06,CUST005,Male,30,Beauty,2,50,100


In [114]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Transaction ID    1000 non-null   int64 
 1   Date              1000 non-null   object
 2   Customer ID       1000 non-null   object
 3   Gender            1000 non-null   object
 4   Age               1000 non-null   int64 
 5   Product Category  1000 non-null   object
 6   Quantity          1000 non-null   int64 
 7   Price per Unit    1000 non-null   int64 
 8   Total Amount      1000 non-null   int64 
dtypes: int64(5), object(4)
memory usage: 70.4+ KB


In [115]:
df.shape

(1000, 9)

In [116]:
df.describe()

,Transaction ID,Age,Quantity,Price per Unit,Total Amount
count,1000.000000,1000.00000,1000.000000,1000.000000,1000.000000
mean,500.500000,41.39200,2.514000,179.890000,456.000000
std,288.819436,13.68143,1.132734,189.681356,559.997632
min,1.000000,18.00000,1.000000,25.000000,25.000000
25%,250.750000,29.00000,1.000000,30.000000,60.000000
50%,500.500000,42.00000,3.000000,50.000000,135.000000
75%,750.250000,53.00000,4.000000,300.000000,900.000000
max,1000.000000,64.00000,4.000000,500.000000,2000.000000


In [117]:
df.isnull().sum()

Transaction ID      0
Date                0
Customer ID         0
Gender              0
Age                 0
Product Category    0
Quantity            0
Price per Unit      0
Total Amount        0
dtype: int64

In [118]:
df.duplicated().sum()

np.int64(0)

In [119]:
df.columns

Index(['Transaction ID', 'Date', 'Customer ID', 'Gender', 'Age',
       'Product Category', 'Quantity', 'Price per Unit', 'Total Amount'],
      dtype='object')

In [120]:
df.corr(numeric_only=True)

,Transaction ID,Age,Quantity,Price per Unit,Total Amount
Transaction ID,1.000000,0.065191,-0.026623,-0.060837,-0.075034
Age,0.065191,1.000000,-0.023737,-0.038423,-0.060568
Quantity,-0.026623,-0.023737,1.000000,0.017501,0.373707
Price per Unit,-0.060837,-0.038423,0.017501,1.000000,0.851925
Total Amount,-0.075034,-0.060568,0.373707,0.851925,1.000000


In [121]:
def clean_data(df):
    df = df.copy()

    # 1. Remove duplicate rows
    df = df.drop_duplicates()

    # 2. Convert Date to datetime
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

    # 3. Handle missing values
    numeric_columns = df.select_dtypes(include="number").columns
    df[numeric_columns] = df[numeric_columns].fillna(0)

    # 4. Remove rows where important values could not be converted
    df = df.dropna(subset=["Date"])

    return df

In [122]:
cleaned_df = clean_data(df)

In [123]:
print("Before:", df.shape)
print("After:", cleaned_df.shape)

Before: (1000, 9)
After: (1000, 9)


In [124]:
cleaned_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Transaction ID    1000 non-null   int64         
 1   Date              1000 non-null   datetime64[ns]
 2   Customer ID       1000 non-null   object        
 3   Gender            1000 non-null   object        
 4   Age               1000 non-null   int64         
 5   Product Category  1000 non-null   object        
 6   Quantity          1000 non-null   int64         
 7   Price per Unit    1000 non-null   int64         
 8   Total Amount      1000 non-null   int64         
dtypes: datetime64[ns](1), int64(5), object(3)
memory usage: 70.4+ KB


In [125]:
calculated_total = cleaned_df["Quantity"] * cleaned_df["Price per Unit"]

(cleaned_df["Total Amount"] != calculated_total).sum()

np.int64(0)

In [126]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv("../.env")

llm = ChatOpenAI(
    model="nvidia/nemotron-3-ultra-550b-a55b:free",
    temperature=0,
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)

response = llm.invoke("Say hello in one short sentence.")

print(response.content)

Hello! How can I help you today?


In [127]:
from typing import TypedDict
import pandas as pd

class AgentState(TypedDict, total=False):
    data: pd.DataFrame
    cleaned_data: pd.DataFrame
    cleaning_summary: str
    analysis: dict
    next_agent: str
    charts: dict

In [128]:
def clean_agent(state: AgentState):
    df = state["data"].copy()

    before_rows = len(df)
    duplicates = df.duplicated().sum()

    # Remove duplicates
    df = df.drop_duplicates()

    # Convert date column
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

    # Remove invalid dates
    invalid_dates = df["Date"].isnull().sum()
    df = df.dropna(subset=["Date"])

    # Fill missing numeric values
    numeric_cols = df.select_dtypes(include="number").columns
    missing_numeric = df[numeric_cols].isnull().sum().sum()

    df[numeric_cols] = df[numeric_cols].fillna(0)

    summary = f"""
Cleaning completed:
- Rows before cleaning: {before_rows}
- Rows after cleaning: {len(df)}
- Duplicates removed: {duplicates}
- Invalid dates removed: {invalid_dates}
- Missing numeric values handled: {missing_numeric}
"""

    return {
        "cleaned_data": df,
        "cleaning_summary": summary
    }

In [129]:
state = {
    "data": df
}

result = clean_agent(state)

cleaned_df = result["cleaned_data"]

print(result["cleaning_summary"])


Cleaning completed:
- Rows before cleaning: 1000
- Rows after cleaning: 1000
- Duplicates removed: 0
- Invalid dates removed: 0
- Missing numeric values handled: 0



In [130]:
cleaned_df.dtypes

Transaction ID               int64
Date                datetime64[ns]
Customer ID                 object
Gender                      object
Age                          int64
Product Category            object
Quantity                     int64
Price per Unit               int64
Total Amount                 int64
dtype: object

In [131]:
def analysis_agent(state: AgentState):
    df = state["cleaned_data"]

    total_sales = df["Total Amount"].sum()
    total_quantity = df["Quantity"].sum()

    category_sales = (
        df.groupby("Product Category")["Total Amount"]
        .sum()
        .sort_values(ascending=False)
    )

    category_quantity = (
        df.groupby("Product Category")["Quantity"]
        .sum()
        .sort_values(ascending=False)
    )

    gender_sales = (
        df.groupby("Gender")["Total Amount"]
        .sum()
        .sort_values(ascending=False)
    )

    top_transactions = df.nlargest(5, "Total Amount")[
        ["Transaction ID", "Product Category", "Quantity",
         "Price per Unit", "Total Amount"]
    ]

    insights = {
        "total_sales": total_sales,
        "total_quantity": total_quantity,
        "category_sales": category_sales,
        "category_quantity": category_quantity,
        "gender_sales": gender_sales,
        "top_transactions": top_transactions
    }

    return {
        "analysis": insights
    }

In [132]:
state = {
    "data": df,
    "cleaned_data": cleaned_df
}

result = analysis_agent(state)

In [133]:
analysis = result["analysis"]

print("Total Sales:", analysis["total_sales"])
print("Total Quantity:", analysis["total_quantity"])

Total Sales: 456000
Total Quantity: 2514


In [134]:
analysis["gender_sales"]

Gender
Female    232840
Male      223160
Name: Total Amount, dtype: int64

In [135]:
from langgraph.graph import StateGraph, START, END

In [136]:
graph = StateGraph(AgentState)

In [137]:
graph.add_node("clean_agent", clean_agent)
graph.add_node("analysis_agent", analysis_agent)

In [138]:
graph.add_edge(START, "clean_agent")
graph.add_edge("clean_agent", "analysis_agent")
graph.add_edge("analysis_agent", END)

In [139]:
app = graph.compile()

In [140]:
initial_state = {
    "data": df
}

final_state = app.invoke(initial_state)

In [141]:
final_state["cleaned_data"].head()

,Transaction ID,Date,Customer ID,Gender,Age,Product Category,Quantity,Price per Unit,Total Amount
0,1,2023-11-24,CUST001,Male,34,Beauty,3,50,150
1,2,2023-02-27,CUST002,Female,26,Clothing,2,500,1000
2,3,2023-01-13,CUST003,Male,50,Electronics,1,30,30
3,4,2023-05-21,CUST004,Male,37,Clothing,1,500,500
4,5,2023-05-06,CUST005,Male,30,Beauty,2,50,100


In [142]:
final_state["analysis"]["total_sales"]

np.int64(456000)

In [143]:
final_state["analysis"]["category_sales"]

Product Category
Electronics    156905
Clothing       155580
Beauty         143515
Name: Total Amount, dtype: int64

In [144]:
def orchestrator_agent(state: AgentState):
    prompt = """
You are the Orchestrator Agent for a retail data pipeline.

Your job is to decide which agent should process the data next.

Available agents:
1. clean_agent - cleans and prepares the raw retail data.
2. analysis_agent - analyzes cleaned retail data.

Rules:
- If the data has not been cleaned, choose clean_agent.
- If the data has been cleaned but not analyzed, choose analysis_agent.
- If analysis is complete, choose end.

Return ONLY one of:
clean_agent
analysis_agent
end
"""

    response = llm.invoke(prompt)

    decision = response.content.strip().lower()

    if "clean_agent" in decision:
        next_agent = "clean_agent"
    elif "analysis_agent" in decision:
        next_agent = "analysis_agent"
    else:
        next_agent = "end"

    return {
        "next_agent": next_agent
    }

In [145]:
response = llm.invoke("Say hello in one short sentence.")
print(response.content)

Hello! How can I help you today?


In [146]:
state = {
    "data": df
}

result = orchestrator_agent(state)

print(result)

{'next_agent': 'clean_agent'}


In [147]:
graph = StateGraph(AgentState)

graph.add_node("orchestrator", orchestrator_agent)
graph.add_node("clean_agent", clean_agent)
graph.add_node("analysis_agent", analysis_agent)

In [148]:
graph.add_edge(START, "orchestrator")

In [149]:
graph.add_conditional_edges(
    "orchestrator",
    lambda state: state["next_agent"],
    {
        "clean_agent": "clean_agent",
        "analysis_agent": "analysis_agent",
        "end": END
    }
)

In [150]:
graph.add_edge("clean_agent", "analysis_agent")
graph.add_edge("analysis_agent", END)

In [151]:
app = graph.compile()

In [152]:
initial_state = {
    "data": df
}

final_state = app.invoke(initial_state)

In [153]:
final_state.keys()

dict_keys(['data', 'cleaned_data', 'cleaning_summary', 'analysis', 'next_agent'])

In [154]:
final_state["cleaning_summary"]

'\nCleaning completed:\n- Rows before cleaning: 1000\n- Rows after cleaning: 1000\n- Duplicates removed: 0\n- Invalid dates removed: 0\n- Missing numeric values handled: 0\n'

In [155]:
final_state["analysis"]["total_sales"]

np.int64(456000)

In [156]:
final_state["analysis"]["category_sales"]

Product Category
Electronics    156905
Clothing       155580
Beauty         143515
Name: Total Amount, dtype: int64

In [157]:
def visualization_agent(state: AgentState):
    df = state["cleaned_data"]
    analysis = state["analysis"]

    os.makedirs("../assets", exist_ok=True)

    # 1. Sales by Category
    plt.figure(figsize=(8, 5))

    analysis["category_sales"].plot(kind="bar")

    plt.title("Sales by Category")
    plt.xlabel("Category")
    plt.ylabel("Total Sales")
    plt.xticks(rotation=0)
    plt.tight_layout()

    category_path = "../assets/sales_by_category.png"
    plt.savefig(category_path)
    plt.close()

    # 2. Sales by Gender
    plt.figure(figsize=(8, 5))

    analysis["gender_sales"].plot(kind="bar")

    plt.title("Sales by Gender")
    plt.xlabel("Gender")
    plt.ylabel("Total Sales")
    plt.xticks(rotation=0)
    plt.tight_layout()

    gender_path = "../assets/sales_by_gender.png"
    plt.savefig(gender_path)
    plt.close()

    # 3. Sales over Time
    daily_sales = (
        df.groupby("Date")["Total Amount"]
        .sum()
        .sort_index()
    )

    plt.figure(figsize=(10, 5))

    daily_sales.plot()

    plt.title("Sales Over Time")
    plt.xlabel("Date")
    plt.ylabel("Total Sales")
    plt.tight_layout()

    time_path = "../assets/sales_over_time.png"
    plt.savefig(time_path)
    plt.close()

    return {
        "charts": {
            "category_sales": category_path,
            "gender_sales": gender_path,
            "sales_over_time": time_path
        }
    }

In [158]:
state = {
    "data": df,
    "cleaned_data": final_state["cleaned_data"],
    "analysis": final_state["analysis"]
}

result = visualization_agent(state)

print(result)

{'charts': {'category_sales': '../assets/sales_by_category.png', 'gender_sales': '../assets/sales_by_gender.png', 'sales_over_time': '../assets/sales_over_time.png'}}


In [159]:
graph = StateGraph(AgentState)

graph.add_node("orchestrator", orchestrator_agent)
graph.add_node("clean_agent", clean_agent)
graph.add_node("analysis_agent", analysis_agent)
graph.add_node("visualization_agent", visualization_agent)

In [160]:
graph.add_edge(START, "orchestrator")

graph.add_conditional_edges(
    "orchestrator",
    lambda state: state["next_agent"],
    {
        "clean_agent": "clean_agent",
        "analysis_agent": "analysis_agent",
        "end": END
    }
)

graph.add_edge("clean_agent", "analysis_agent")
graph.add_edge("analysis_agent", "visualization_agent")
graph.add_edge("visualization_agent", END)

In [161]:
app = graph.compile()

In [162]:
initial_state = {
    "data": df
}

final_state = app.invoke(initial_state)

In [163]:
final_state.keys()

dict_keys(['data', 'cleaned_data', 'cleaning_summary', 'analysis', 'next_agent', 'charts'])

In [164]:
final_state["charts"]

{'category_sales': '../assets/sales_by_category.png',
 'gender_sales': '../assets/sales_by_gender.png',
 'sales_over_time': '../assets/sales_over_time.png'}